Author: Bryan Sandor

Course: Stat 554

Title: Final Project

# Preamble

For this project, we'll use a `Jupyter` notebook fitting a machine learning model using `pyspark`'s `MLib` module. We'll use code to read in a stream of data and use that model to perform predictions on the stream, outputing them to the console. The data is modified from the UCI machine learning repository and studies the relationship between power consumption in Tetouan City and various factors like time of day, temperature, and humidity.

We begin by importing all the necessary libraries.

In [1]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.ml.feature import SQLTransformer, Binarizer, OneHotEncoder, VectorAssembler, PCA
from pyspark.ml.regression import LinearRegression
from pyspark.ml import Pipeline
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import RegressionEvaluator
import pyspark.sql.functions as F
import time

# Part 1: Fitting the Model

## Reading in the Data

We will read the data in using a `pandas` data frame, initially.

In [2]:
powerData = pd.read_csv("power_ml_data.csv")

Then we convert it to a `spark` data frame, first initializing our `spark` session.

In [3]:
spark = SparkSession.builder \
        .appName("proj2") \
        .config("spark.sql.ansi.enabled", "false") \
        .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/05 17:44:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
powerDF = spark.createDataFrame(powerData)
powerDF.show(n = 10, truncate = False)

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|6.559      |73.8    |0.083     |0.051                |0.119        |34055.6962  |16128.87538 |20240.96386 |1    |0   |
|6.414      |74.5    |0.083     |0.07                 |0.085        |29814.68354 |19375.07599 |20131.08434 |1    |0   |
|6.313      |74.5    |0.08      |0.062                |0.1          |29128.10127 |19006.68693 |19668.43373 |1    |0   |
|6.121      |75.0    |0.083     |0.091                |0.096        |28228.86076 |18361.09422 |18899.27711 |1    |0   |
|5.921      |75.7    |0.081     |0.048                |0.085        |27335.6962  |17872.34043 |18442.40964 |1    |0   |
|5.853      |76.9    |0.081     |0.059  

For this assignment, we will use `Power_Zone_3` as the response and the other variables as predictors.

## Pipeline: Manipulations and Formatting

Now we need to make some adjustments to the data.

### Change `Hour` Class

In [5]:
powerDF.schema

StructType([StructField('Temperature', DoubleType(), True), StructField('Humidity', DoubleType(), True), StructField('Wind_Speed', DoubleType(), True), StructField('General_Diffuse_Flows', DoubleType(), True), StructField('Diffuse_Flows', DoubleType(), True), StructField('Power_Zone_1', DoubleType(), True), StructField('Power_Zone_2', DoubleType(), True), StructField('Power_Zone_3', DoubleType(), True), StructField('Month', LongType(), True), StructField('Hour', LongType(), True)])

First, we need to re-cast the `Hour` column as `DoubleType` instead of `LongType()`.

In [6]:
recast = SQLTransformer(statement="SELECT *, CAST(Hour AS DOUBLE) AS db_hour FROM __THIS__")

recast.transform(powerDF).schema

StructType([StructField('Temperature', DoubleType(), True), StructField('Humidity', DoubleType(), True), StructField('Wind_Speed', DoubleType(), True), StructField('General_Diffuse_Flows', DoubleType(), True), StructField('Diffuse_Flows', DoubleType(), True), StructField('Power_Zone_1', DoubleType(), True), StructField('Power_Zone_2', DoubleType(), True), StructField('Power_Zone_3', DoubleType(), True), StructField('Month', LongType(), True), StructField('Hour', LongType(), True), StructField('db_hour', DoubleType(), True)])

### Binarize `Hour` Column

Now that the `Hour` column is appropriately cast, we binarize it with the margin at $6.5$, i.e., night vs day.

In [7]:
binaryTrans = Binarizer(threshold = 6.5, inputCol = "db_hour", outputCol = "bin_hour")

binaryTrans.transform(
    recast.transform(powerDF)
).show()

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+-------+--------+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|db_hour|bin_hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+-------+--------+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|    0.0|     0.0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|    0.0|     0.0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|    0.0|     0.0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422| 18899.27711|    1|   0|    0.0|     0.0|
|      5.921|    75.7|     0.081|        

### One-Hot Encode the `Month` Column

Next, we will one-hot encode the `Month` column

In [8]:
encodeTrans = OneHotEncoder(inputCol = "Month", outputCol = "enc_month")

encodeTrans.fit(powerDF).transform(
    binaryTrans.transform(
        recast.transform(powerDF)
    )
).show()

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+-------+--------+--------------+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|db_hour|bin_hour|     enc_month|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+-------+--------+--------------+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|    0.0|     0.0|(12,[1],[1.0])|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|    0.0|     0.0|(12,[1],[1.0])|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|    0.0|     0.0|(12,[1],[1.0])|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361

### PCA fit

To perform a PCA fit, we need to map several columns into a single column, which we will accomplish using the `VectorAssembler()`, combining `Temperature`, `Humidity`, `Wind_Speed`, `General_Diffuse_Flows`, and `Diffuse_Flows`.

In [9]:
assemblerPCA = VectorAssembler(inputCols = ["Temperature", "Humidity", "Wind_Speed", "General_Diffuse_Flows", "Diffuse_Flows"], outputCol = "PC", handleInvalid = "skip")

In [10]:
assemblerPCA.transform(
    encodeTrans.fit(powerDF).transform(
    binaryTrans.transform(
        recast.transform(powerDF)
    )
    )
).show()

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+-------+--------+--------------+--------------------+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|db_hour|bin_hour|     enc_month|                  PC|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+-------+--------+--------------+--------------------+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|    0.0|     0.0|(12,[1],[1.0])|[6.559,73.8,0.083...|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|    0.0|     0.0|(12,[1],[1.0])|[6.414,74.5,0.083...|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|    0.0|     0.0

Now we perform the PCA fit.

In [11]:
pca = PCA(k = 2, inputCol = "PC")
pca.setOutputCol("pca_features")
pcaTrans = pca.fit(
    assemblerPCA.transform(
    encodeTrans.fit(powerDF).transform(
    binaryTrans.transform(
        recast.transform(powerDF)
    )
    )
    )
)

In [12]:
pcaTrans.transform(
    assemblerPCA.transform(
    encodeTrans.fit(powerDF).transform(
    binaryTrans.transform(
        recast.transform(powerDF)
    )
    )
    )
).show()

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+-------+--------+--------------+--------------------+--------------------+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|db_hour|bin_hour|     enc_month|                  PC|        pca_features|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+-------+--------+--------------+--------------------+--------------------+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|    0.0|     0.0|(12,[1],[1.0])|[6.559,73.8,0.083...|[1.79440486365695...|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|    0.0|     0.0|(12,[1],[1.0])|[6.414,74.5,0.083...|[1.80604083009822...|
|      6.313|    74.5|      0.

### Response and Predictors

To prepare the data for analysis, we begin by setting `Power_Zone_3` as our response by renaming it `label`.

In [13]:
labelTrans = SQLTransformer(
    statement = """
                SELECT pca_features, bin_hour, Power_Zone_1, Power_Zone_2, enc_month, Power_Zone_3 as label FROM __THIS__
                """
)

labelTrans.transform(
    pcaTrans.transform(
    assemblerPCA.transform(
    encodeTrans.fit(powerDF).transform(
    binaryTrans.transform(
        recast.transform(powerDF)
    )
    )
    )
    )
).show()

+--------------------+--------+------------+------------+--------------+-----------+
|        pca_features|bin_hour|Power_Zone_1|Power_Zone_2|     enc_month|      label|
+--------------------+--------+------------+------------+--------------+-----------+
|[1.79440486365695...|     0.0|  34055.6962| 16128.87538|(12,[1],[1.0])|20240.96386|
|[1.80604083009822...|     0.0| 29814.68354| 19375.07599|(12,[1],[1.0])|20131.08434|
|[1.81022976305638...|     0.0| 29128.10127| 19006.68693|(12,[1],[1.0])|19668.43373|
|[1.79866765174088...|     0.0| 28228.86076| 18361.09422|(12,[1],[1.0])|18899.27711|
|[1.86328720163796...|     0.0|  27335.6962| 17872.34043|(12,[1],[1.0])|18442.40964|
|[1.87820674500460...|     0.0| 26624.81013| 17416.41337|(12,[1],[1.0])|18130.12048|
|[1.91529298717955...|     0.0| 25998.98734| 16993.31307|(12,[1],[1.0])|17945.06024|
|[1.92400540807028...|     0.0| 25446.07595| 16661.39818|(12,[1],[1.0])|17459.27711|
|[1.89501930353027...|     0.0| 24777.72152| 16227.35562|(12,[1],

Finally, we combine the predictors into a single column named `features`.

In [14]:
assemblerFeatures = VectorAssembler(inputCols = ["pca_features", "bin_hour", "Power_Zone_1", "Power_Zone_2", "enc_month"], outputCol = "features")

assemblerFeatures.transform(
    labelTrans.transform(
    pcaTrans.transform(
    assemblerPCA.transform(
    encodeTrans.fit(powerDF).transform(
    binaryTrans.transform(
        recast.transform(powerDF)
    )
    )
    )
    )
    )
).show(truncate = False)

+----------------------------------------+--------+------------+------------+--------------+-----------+-------------------------------------------------------------------------------------+
|pca_features                            |bin_hour|Power_Zone_1|Power_Zone_2|enc_month     |label      |features                                                                             |
+----------------------------------------+--------+------------+------------+--------------+-----------+-------------------------------------------------------------------------------------+
|[1.7944048636569516,-0.7412746447404612]|0.0     |34055.6962  |16128.87538 |(12,[1],[1.0])|20240.96386|(17,[0,1,3,4,6],[1.7944048636569516,-0.7412746447404612,34055.6962,16128.87538,1.0]) |
|[1.8060408300982287,-0.7108534239558637]|0.0     |29814.68354 |19375.07599 |(12,[1],[1.0])|20131.08434|(17,[0,1,3,4,6],[1.8060408300982287,-0.7108534239558637,29814.68354,19375.07599,1.0])|
|[1.8102297630563877,-0.7283113191159094]|0.0

## Elastic Net Model

Given specific parameters, we will use cross validation and linear regression to fit an elastic net model with our previous transformations in a pipeline.

In [15]:
lr = LinearRegression(standardization = True)

paramGrid = ParamGridBuilder() \
    .addGrid(lr.regParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .addGrid(lr.elasticNetParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .build()

pipeline_lm = Pipeline(stages = [recast, binaryTrans, encodeTrans, assemblerPCA, pcaTrans, labelTrans, assemblerFeatures, lr])

At the suggestion of a classmate, we will make use of the parallel processing for a $5$-fold cross validation (using the default root mean squared error (RMSE)).

In [16]:
crossval_lm = CrossValidator(estimator = pipeline_lm,
                             estimatorParamMaps = paramGrid,
                             evaluator = RegressionEvaluator(),
                             parallelism = 128,
                             seed = 1982,
                             numFolds = 5)

Finally we fit the model.

In [19]:
cvLinModel = crossval_lm.fit(powerDF)

26/05/05 17:49:43 ERROR LBFGS: Failure! Resetting history: breeze.optimize.FirstOrderException: Line search zoom failed
26/05/05 17:49:43 ERROR LBFGS: Failure again! Giving up and returning. Maybe the objective is just poorly behaved?
26/05/05 17:49:45 ERROR LBFGS: Failure! Resetting history: breeze.optimize.FirstOrderException: Line search zoom failed


(The author admits he had to run the above line twice to get it to process correctly.) We print out all $11 \times 11 = 121$ combinations.

In [20]:
my_list = []
for i in range(len(paramGrid)):
    my_list.append([cvLinModel.avgMetrics[i], paramGrid[i].values()])
my_list

[[np.float64(2147.547722631705), dict_values([0.0, 0.0])],
 [np.float64(2147.547722631706), dict_values([0.0, 0.05])],
 [np.float64(2147.547722631704), dict_values([0.0, 0.1])],
 [np.float64(2147.547723081113), dict_values([0.0, 0.25])],
 [np.float64(2147.5477232063176), dict_values([0.0, 0.5])],
 [np.float64(2147.5477226317043), dict_values([0.0, 0.75])],
 [np.float64(2147.5477230184006), dict_values([0.0, 0.9])],
 [np.float64(2147.5477222923246), dict_values([0.0, 0.95])],
 [np.float64(2147.547722631704), dict_values([0.0, 0.98])],
 [np.float64(2147.547722631705), dict_values([0.0, 0.99])],
 [np.float64(2147.5477226317057), dict_values([0.0, 1.0])],
 [np.float64(2147.5477414063575), dict_values([0.05, 0.0])],
 [np.float64(2147.5482890271105), dict_values([0.05, 0.05])],
 [np.float64(2147.5473932101586), dict_values([0.05, 0.1])],
 [np.float64(2147.5477102047225), dict_values([0.05, 0.25])],
 [np.float64(2147.5481244527728), dict_values([0.05, 0.5])],
 [np.float64(2147.546907066976), 

Then we seek the parameters for the best model.

In [21]:
bestLModel = cvLinModel.bestModel.stages[-1]
bestLModel.extractParamMap()

{Param(parent='LinearRegression_ea533aa2ce6d', name='aggregationDepth', doc='suggested depth for treeAggregate (>= 2).'): 2,
 Param(parent='LinearRegression_ea533aa2ce6d', name='elasticNetParam', doc='the ElasticNet mixing parameter, in range [0, 1]. For alpha = 0, the penalty is an L2 penalty. For alpha = 1, it is an L1 penalty.'): 0.25,
 Param(parent='LinearRegression_ea533aa2ce6d', name='epsilon', doc='The shape parameter to control the amount of robustness. Must be > 1.0. Only valid when loss is huber'): 1.35,
 Param(parent='LinearRegression_ea533aa2ce6d', name='featuresCol', doc='features column name.'): 'features',
 Param(parent='LinearRegression_ea533aa2ce6d', name='fitIntercept', doc='whether to fit an intercept term.'): True,
 Param(parent='LinearRegression_ea533aa2ce6d', name='labelCol', doc='label column name.'): 'label',
 Param(parent='LinearRegression_ea533aa2ce6d', name='loss', doc='The loss function to be optimized. Supported options: squaredError, huber.'): 'squaredErro

The key lines in this printout concern the `elasticNetParam`, which for the best model is $\alpha = 0.25$, and the `regParam`, which is $\lambda = 0.1$. We then find the error associated with these parameters, ie., the minimum error through cross validation.

In [22]:
CVError = min(cvLinModel.avgMetrics)
CVError

np.float64(2147.5468555075267)

This may be verified by parsing the above table, most easily done via `Ctrl+F`. Using the entire dataset as the training set we perform predictions using the model.

In [23]:
cvLinModel.transform(powerDF).select("label", "prediction").show()

+-----------+------------------+
|      label|        prediction|
+-----------+------------------+
|20240.96386| 20880.33970130768|
|20131.08434| 18659.92147808184|
|19668.43373|18204.427090265952|
|18899.27711|17590.347680927844|
|18442.40964|16996.978968117986|
|18130.12048| 16517.36635942665|
|17945.06024|16092.934692961655|
|17459.27711|15722.379081012878|
|17025.54217|15270.731694846698|
|16794.21687|14938.028946615854|
|16638.07229|14652.143575261834|
|16395.18072|14414.660768824851|
|16117.59036|14082.559411468657|
| 15822.6506|13624.562955558995|
|15672.28916|13450.063852264448|
|15597.10843|13302.029790063458|
|15510.36145|13154.609299214131|
|15336.86747|12994.976789618238|
|15140.24096| 12758.66919896612|
|15059.27711|12674.752406059499|
+-----------+------------------+
only showing top 20 rows


And then we find the RMSE for the model.

In [24]:
train_error_lm = RegressionEvaluator().evaluate(cvLinModel.transform(powerDF))
print(train_error_lm)

2147.097381290536


Lastly, we compare the actual values, the predicted values, and examine their residuals (difference between them).

In [25]:
resultsPowerDF = cvLinModel.transform(powerDF).withColumn("residuals", F.col("label") - F.col("prediction"))

In [26]:
resultsPowerDF.select("label", "prediction", "residuals").show()

+-----------+------------------+------------------+
|      label|        prediction|         residuals|
+-----------+------------------+------------------+
|20240.96386| 20880.33970130768|-639.3758413076794|
|20131.08434| 18659.92147808184| 1471.162861918161|
|19668.43373|18204.427090265952|1464.0066397340488|
|18899.27711|17590.347680927844|1308.9294290721555|
|18442.40964|16996.978968117986|1445.4306718820153|
|18130.12048| 16517.36635942665|1612.7541205733505|
|17945.06024|16092.934692961655| 1852.125547038344|
|17459.27711|15722.379081012878|1736.8980289871215|
|17025.54217|15270.731694846698|1754.8104751533028|
|16794.21687|14938.028946615854|1856.1879233841464|
|16638.07229|14652.143575261834|1985.9287147381656|
|16395.18072|14414.660768824851| 1980.519951175149|
|16117.59036|14082.559411468657|2035.0309485313428|
| 15822.6506|13624.562955558995| 2198.087644441006|
|15672.28916|13450.063852264448|2222.2253077355526|
|15597.10843|13302.029790063458|2295.0786399365425|
|15510.36145

# Part Two: Streaming Data

For this portion, we are going to stream in `.csv` files.

## Reading a Stream

We begin by defining the schema from the source file.

In [27]:
mySchema = spark.read.csv('power_streaming_data.csv', header = True, inferSchema = True).schema

In [28]:
mySchema

StructType([StructField('Temperature', DoubleType(), True), StructField('Humidity', DoubleType(), True), StructField('Wind_Speed', DoubleType(), True), StructField('General_Diffuse_Flows', DoubleType(), True), StructField('Diffuse_Flows', DoubleType(), True), StructField('Power_Zone_1', DoubleType(), True), StructField('Power_Zone_2', DoubleType(), True), StructField('Power_Zone_3', DoubleType(), True), StructField('Month', IntegerType(), True), StructField('Hour', IntegerType(), True)])

## Transform/Aggregation Step

We begin by initiating the first of two transformations. In this first transformation, we use the previous model to obtain predictions from the incoming data. The `readStream` will "watch" a folder named "finalProjStream" for new `.csv` files.

In [29]:
streamDF = spark.readStream.schema(mySchema).csv("finalProjStream")

streamPowerDF = cvLinModel.transform(streamDF).withColumn("residuals", F.col("label") - F.col("prediction")).select("label", "prediction", "residuals")

The second transformation will act on that same `.csv` file, changing the column `Power_Zone_3` to `label` so that after the two transformations, the two dataframes will share a common column.

In [30]:
labelTrans2 = SQLTransformer(
    statement = """
                SELECT Temperature, Humidity, Wind_Speed, General_Diffuse_Flows, Diffuse_Flows, Power_Zone_1, Power_Zone_2, Month, Hour, Power_Zone_3 as label FROM __THIS__
                """
)

streamPowerDF2 = labelTrans2.transform(streamDF)

## Writing Step

We will join the two dataframes on that common column, `label`, and write the output to the console.

In [31]:
joinquery = streamPowerDF \
                .join(streamPowerDF2, on = "label", how = "inner") \
                .writeStream.outputMode("append") \
                .format("console") \
                .start()

-------------------------------------------
Batch: 0
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|          residuals|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|17447.71084|17666.980753222124|-219.26991322212416|       8.49|    76.9|     0.083|                0.084|        0.141| 28362.53165| 18320.97264|    1|   0|
|16029.58794| 16727.47835328348| -697.8904132834814|      16.32|   52.15|     0.083|                603.7|        160.5| 32570.84746| 20677.20365|    2|  11|
|15381.29032|16410.424134903264|-1029.1338149032636|      15.49|   63.14|     0.083|                324.0|       

In [32]:
joinquery.stop()

# Part Three: Produce Data

## Not Live Portion

For the last portion of this project, we initiate another `writeStream` to "watch" the appropriate folder, `finalProjStream`. Then, we write the attached `python` code in the console, to randomly sample $5$ rows from our data set and write those to the "watched" folder, pausing for $20$ seconds inbetween sampling. We do this $20$ times and then terminate the `writeStream`.

In [33]:
joinquery = streamPowerDF \
                .join(streamPowerDF2, on = "label", how = "inner") \
                .writeStream.outputMode("append") \
                .format("console") \
                .start()

-------------------------------------------
Batch: 0
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|          residuals|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|15115.63636|14934.938776398114| 180.69758360188644|      14.73|    87.5|     0.076|                0.033|        0.126| 23052.40043|  12328.7169|    4|   2|
|10113.83044| 7812.827951394228| 2301.0024886057718|      20.66|   55.99|     4.922|                0.117|        0.044| 20191.85841| 11492.30769|    9|   6|
|11975.19757|12433.432840873927| -458.2352708739272|       25.4|   62.02|     4.922|                520.2|       

-------------------------------------------
Batch: 1
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|          residuals|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|16755.30364|16210.716341864947|  544.5872981350512|      21.56|    70.2|     4.926|                637.8|        592.3| 34068.98361| 22183.28173|    5|  16|
|    25036.8|26621.503594848185|-1584.7035948481862|      19.76|   65.59|      0.08|                0.062|        0.107| 41032.05298| 22075.25988|    6|   2|
|12532.04819|11448.552753504013| 1083.4954364959867|      13.59|   60.73|     4.922|                124.8|       

-------------------------------------------
Batch: 2
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|          residuals|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|  29209.279|26654.514168875296|  2554.764831124703|      28.62|   63.79|     4.903|                231.5|        187.3| 40473.42952|  28343.8226|    8|  17|
|14388.43373|13377.711616324377| 1010.7221136756234|      16.96|   60.88|     4.918|                0.055|        0.115| 22152.91139| 13415.19757|    1|   5|
|19287.53799| 20338.75677843699|-1051.2187884369887|      20.91|   66.12|     4.916|                0.091|       

-------------------------------------------
Batch: 3
-------------------------------------------
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|         residuals|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|16649.63855|17595.077221920248|-945.4386719202485|       8.74|    76.9|     0.085|                329.2|        343.5| 33843.03797|  18955.6231|    1|  10|
| 7610.60241| 9199.072068260844|-1588.469658260844|      15.78|   63.63|      0.08|                 84.3|        21.27| 24116.92308| 19476.44628|   11|   8|
|32646.01881|  32277.3731064493|  368.645703550701|      23.31|   62.12|      4.92|                0.091|        0.082

-------------------------------------------
Batch: 4
-------------------------------------------
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|         residuals|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|11034.21687| 9082.898005511837| 1951.318864488163|      18.64|   69.05|     0.068|                0.077|        0.122| 21150.76923| 16475.20661|   11|   6|
|27036.48903| 26150.55422605002| 885.9348039499819|      27.25|    71.7|     4.906|                361.4|        250.9| 40332.78579| 28777.19113|    8|  16|
|10224.57831|13075.206313062623|-2850.628003062622|      19.25|   60.96|     0.075|                341.8|        40.34

...

-------------------------------------------
Batch: 15
-------------------------------------------
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|         residuals|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|12603.38558|18293.597180360965|-5690.211600360964|       19.9|   45.75|     4.908|                 0.11|        0.133| 23819.93341|  18368.7434|    8|   6|
|12421.81818|14580.826971942817|-2159.008791942817|      10.76|    78.4|     0.081|                45.33|        40.22| 25135.67277| 15646.43585|    4|   7|
|14561.92771| 13822.39301906191| 739.5346909380896|       10.8|    80.6|     0.085|                0.026|        0.17

-------------------------------------------
Batch: 16
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|          residuals|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|8303.481393| 7974.950414651915|  328.5309783480852|      17.67|   53.74|     0.082|                0.055|        0.145| 22320.91255| 18237.49616|   12|   6|
|    22963.2|25250.899666690337| -2287.699666690336|       20.4|    80.5|     0.079|                0.029|        0.133| 38845.03311| 21641.16424|    6|   3|
|10912.10031|15769.051131444572| -4856.950821444572|      21.07|    72.8|     4.924|                 95.0|      

-------------------------------------------
Batch: 17
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|          residuals|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|11508.34008|13442.707868374146|-1934.3677883741457|      17.89|    79.8|     4.916|                1.046|        0.949| 22750.42623| 15161.60991|    5|   6|
|28439.27273|25754.153664918707| 2685.1190650812932|      16.73|    70.9|     0.082|                0.037|        0.119| 42149.06351| 21735.64155|    4|  21|
|12387.46988|12404.035368727895|-16.565488727894262|      19.41|   68.28|     0.086|                377.0|      

-------------------------------------------
Batch: 18
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|          residuals|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|9882.352941|11542.436671766907| -1660.083730766908|      16.56|   58.08|     0.087|                209.9|         93.1| 31063.11787| 25259.28199|   12|  13|
|9283.073229|11027.622735056975|-1744.5495060569756|      10.62|   68.31|     0.086|                428.9|        36.11| 30856.27376| 24559.68088|   12|  11|
| 10769.7479| 10092.95717224582|  676.7907277541799|      13.41|    72.0|     0.085|                0.066|      

-------------------------------------------
Batch: 19
-------------------------------------------
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|         residuals|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|9979.331307| 9451.236288661541| 528.0950183384593|      21.22|   57.94|     0.085|                45.88|        29.02| 27085.86433|  21121.9917|   10|   8|
| 9611.52461| 7368.561504641458|2242.9631053585426|      14.84|   58.85|     0.078|                 0.04|        0.104| 21450.95057| 17493.70973|   12|   2|
|9566.659857| 8406.569773144252|1160.0900838557482|      19.61|    74.2|     0.338|                0.117|        0.14

In [34]:
joinquery.stop()

## Live Portion

We repeat this process, "live," one more time to round out the project.

In [38]:
joinquery = streamPowerDF \
                .join(streamPowerDF2, on = "label", how = "inner") \
                .writeStream.outputMode("append") \
                .format("console") \
                .start()

-------------------------------------------
Batch: 0
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|          residuals|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|15519.35223| 15557.09681193221| -37.74458193220926|       23.4|   66.88|     4.915|                847.0|        193.5| 32243.40984| 20017.33746|    5|  11|
|13696.77812|10747.892731016855|  2948.885388983146|      20.66|   68.97|     0.082|                0.051|        0.096| 28610.94092|  22884.6473|   10|  23|
|13256.68342|14515.878577032523|-1259.1951570325236|      11.15|    84.0|     4.916|                216.7|       

-------------------------------------------
Batch: 1
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|          residuals|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|14128.19277|16826.663447599636|-2698.4706775996365|      12.82|   69.49|     0.082|                111.9|        114.9| 30610.63291| 19524.62006|    1|  10|
|15307.63636|15184.730068295674| 122.90629170432658|      14.81|    88.7|     0.071|                0.026|        0.108| 23684.82239| 11085.94705|    4|   5|
|     9600.0|10649.932297559146|-1049.9322975591458|      14.27|   43.52|     0.079|                238.0|       

-------------------------------------------
Batch: 2
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|          residuals|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|17308.91566| 18437.18530692286|-1128.2696469228613|      14.65|   67.51|     0.083|                462.5|         88.7|  34043.5443| 21589.05775|    1|  12|
|23331.49798|23037.400104447755| 294.09787555224466|      20.09|    62.8|     4.916|                38.51|         33.1| 40087.08197| 24713.31269|    5|  19|
|31866.77824| 25730.48890598404|  6136.289334015961|      31.69|   35.18|     4.908|                711.0|       

In [39]:
joinquery.stop()